# 060 — Round 4: grouped k-fold training (`attention_unet_nll`)

`fixing.md` #4. A genuine held-out **variance** estimate — how much the metrics move
across independent train/val splits — instead of the single fixed split every other
notebook uses.

**Scope** (decided from `C4`/`C5`, `final-comments.md`): **one model**,
`attention_unet_nll` — top-2 on every detection cut and the NLL head covers both
`structural delta` (from its `μ`) and `structural z`. `k = settings.KFOLD_K` (3).
`β = settings.NLL_BETA` (0.5), fixed. A second model is only added if fold variance
turns out alarming.

**Split** (`scripts.kfold`): real artworks partitioned into `k` groups *by artwork
ID* (no section leakage); fold `i` holds out group `i` as validation, the rest **plus
all mockups** are train. No test split — the held-out artworks are the fold's
evaluation set. `data/test/` (GT paintings) is untouched.

**Run it across several short sessions.** Each fold is one `train_single.py`
subprocess that exits when done; the loop below is **skip-if-exists**, so re-running
this notebook trains only the next missing fold. Set `MAX_FOLDS_PER_RUN = 1` to stop
after one fold per session. Realistic per fold on this hardware: ~1–3 h (the NLL runs
in `024` early-stopped around epoch 25; the 100-epoch cap is the upper bound).

Evaluate with `061_kfold_evaluation.ipynb` once ≥ 2 folds exist.

Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports.

In [ ]:
import json
import subprocess

from scripts.config import settings
from scripts.dataset import load_image_pairs
from scripts.kfold import fold_artwork_groups, grouped_kfold_splits

print(f"KFOLD_K = {settings.KFOLD_K}   KFOLD_SEED = {settings.KFOLD_SEED}   NLL_BETA = {settings.NLL_BETA}")

## 1. The fold plan

Deterministic in `(KFOLD_K, KFOLD_SEED)` and the set of real-artwork IDs. Mockups are
in every fold's train set.

In [ ]:
ARCH = "attention_unet_nll"
K = settings.KFOLD_K
KFOLD_DIR = settings.MODELS_DIR / "kfold"
KFOLD_LOG_DIR = settings.LOGS_DIR / "kfold"

pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
held_out = fold_artwork_groups(pairs, k=K, seed=settings.KFOLD_SEED)
splits = grouped_kfold_splits(pairs, k=K, seed=settings.KFOLD_SEED)

print(f"{len(pairs)} pairs total\n")
for i, (groups, (tr, va)) in enumerate(zip(held_out, splits)):
    print(f"fold {i}: held out {len(groups)} artworks -> val {len(va)} pairs / train {len(tr)} pairs")
    print(f"         {groups}")

## 2. Train — one subprocess per fold, resumable

`scripts.train_single --fold i --kfold-k K` builds fold `i`'s split itself
(deterministic from `settings`), so nothing crosses the process boundary. Checkpoints
land in `models/kfold/fold_<i>/attention_unet_nll/best_model.keras` (+ `history.json`).

A fold whose checkpoint already exists is skipped. `MAX_FOLDS_PER_RUN` caps how many
folds this run will train — set it to `1` to do exactly one fold per session.

In [ ]:
MAX_FOLDS_PER_RUN = None  # e.g. 1 to train a single fold per session
EPOCHS = settings.EPOCHS

trained_this_run = 0
for fold in range(K):
    model_dir = KFOLD_DIR / f"fold_{fold}"
    ckpt = model_dir / ARCH / "best_model.keras"
    if ckpt.exists():
        print(f"[skip] fold {fold} — checkpoint at {ckpt}")
        continue
    if MAX_FOLDS_PER_RUN is not None and trained_this_run >= MAX_FOLDS_PER_RUN:
        print(f"[stop] MAX_FOLDS_PER_RUN={MAX_FOLDS_PER_RUN} reached — re-run to continue")
        break

    print(f"\n{'=' * 60}\n  training fold {fold}/{K}\n{'=' * 60}")
    cmd = [
        sys.executable, "-m", "scripts.train_single",
        "--arch", ARCH,
        "--epochs", str(EPOCHS),
        "--model-dir", str(model_dir),
        "--log-dir", str(KFOLD_LOG_DIR / f"fold_{fold}"),
        "--nll",
        "--loss-name", "laplace_nll",
        "--nll-beta", str(settings.NLL_BETA),
        "--fold", str(fold),
        "--kfold-k", str(K),
    ]
    subprocess.run(cmd, cwd=project_root, check=True)
    trained_this_run += 1

    hist = json.loads((model_dir / ARCH / "history.json").read_text())
    print(f"\nfold {fold}: best val_loss = {min(hist['val_loss']):.4f}  "
          f"({len(hist['val_loss'])} epochs)")

print(f"\ntrained {trained_this_run} fold(s) this run")

## 3. Status

In [ ]:
print(f"{'fold':<6}{'checkpoint':<12}{'epochs':<9}{'best val_loss':<14}held-out artworks")
print("-" * 90)
for fold in range(K):
    hp = KFOLD_DIR / f"fold_{fold}" / ARCH / "history.json"
    ck = (KFOLD_DIR / f"fold_{fold}" / ARCH / "best_model.keras").exists()
    if hp.exists():
        h = json.loads(hp.read_text())
        ep, bvl = len(h["val_loss"]), min(h["val_loss"])
        print(f"{fold:<6}{'ok' if ck else 'MISSING':<12}{ep:<9}{bvl:<14.4f}{held_out[fold]}")
    else:
        print(f"{fold:<6}{'-':<12}{'-':<9}{'-':<14}{held_out[fold]}")

done = sum((KFOLD_DIR / f'fold_{f}' / ARCH / 'best_model.keras').exists() for f in range(K))
print(f"\n{done}/{K} folds trained." + ("  -> run 061_kfold_evaluation.ipynb" if done >= 2 else "  -> train more before evaluating"))